# AWS S3 Operations with boto3
All operations use the local folder `C:\projects\aws\Sample` for uploads/downloads.

In [11]:
import boto3
import os

# S3 client — uses credentials from ~/.aws/credentials or environment variables
s3 = boto3.client('s3', region_name='ap-south-2')

# Base local directory used for all upload/download operations
LOCAL_DIR = r'C:\projects\aws\Sample'

# S3 bucket name — must be globally unique across all AWS accounts
BUCKET_NAME = 'kousik-bucket-2026'

print("boto3 S3 client ready.")

boto3 S3 client ready.


## 1. Create an S3 Bucket
Creates a new S3 bucket in the specified region.

In [12]:
s3.create_bucket(
    Bucket=BUCKET_NAME,                                      # Globally unique bucket name
    CreateBucketConfiguration={'LocationConstraint': 'ap-south-2'}  # Required for all regions except us-east-1
)
print(f"Bucket '{BUCKET_NAME}' created successfully.")

Bucket 'kousik-bucket-2026' created successfully.


## 2. Upload a File to S3
Uploads a single local file to the S3 bucket.

In [13]:
local_file = os.path.join(LOCAL_DIR, 'file4')   # Full path to the local file
s3_key     = 'file4'                             # Key (path/name) the file will have in S3

s3.upload_file(
    Filename=local_file,   # Local file path to upload
    Bucket=BUCKET_NAME,    # Target S3 bucket
    Key=s3_key             # Destination key inside the bucket
)
print(f"Uploaded '{local_file}' → s3://{BUCKET_NAME}/{s3_key}")

Uploaded 'C:\projects\aws\Sample\file4' → s3://kousik-bucket-2026/file4


## 3. List Objects/Files in an S3 Bucket
Lists all object keys (file paths) stored in the bucket.

In [14]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME)  # list_objects_v2 is preferred over list_objects

if 'Contents' not in response:
    print("Bucket is empty.")
else:
    for obj in response['Contents']:
        print(f"  Key: {obj['Key']:<40}  Size: {obj['Size']} bytes  Modified: {obj['LastModified']}")

  Key: file4                                     Size: 0 bytes  Modified: 2026-05-18 14:24:37+00:00


## 4. Download a File from S3 to Local
Downloads a single S3 object to the local `Sample` directory.

In [15]:
s3_key         = 'file4'                                        # Key of the object to download from S3
download_path  = os.path.join(LOCAL_DIR, 'file4_downloaded')    # Local destination path

s3.download_file(
    Bucket=BUCKET_NAME,    # Source S3 bucket
    Key=s3_key,            # S3 object key to download
    Filename=download_path # Local path where the file will be saved
)
print(f"Downloaded s3://{BUCKET_NAME}/{s3_key} → '{download_path}'")

Downloaded s3://kousik-bucket-2026/file4 → 'C:\projects\aws\Sample\file4_downloaded'


## 5. Upload All Files in a Directory to S3
Recursively uploads every file inside `C:\projects\aws\Sample` preserving folder structure as S3 keys.

In [16]:
for root, dirs, files in os.walk(LOCAL_DIR):       # Walk all subdirectories recursively
    for filename in files:
        local_path = os.path.join(root, filename)

        # Build S3 key by making path relative to LOCAL_DIR and converting \ to /
        relative_path = os.path.relpath(local_path, LOCAL_DIR)
        s3_key = relative_path.replace('\\', '/')  # S3 keys use forward slashes

        s3.upload_file(
            Filename=local_path,   # Full local file path
            Bucket=BUCKET_NAME,    # Target S3 bucket
            Key=s3_key             # Relative path becomes the S3 key
        )
        print(f"Uploaded: {s3_key}")

Uploaded: file4
Uploaded: folder1/file1
Uploaded: folder2/file2
Uploaded: folder3/file3


## 6. Download Full S3 Directory to Local
Downloads all objects from the bucket, recreating the folder structure inside `C:\projects\aws\Sample`.

In [17]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME)  # Get all objects in the bucket

for obj in response.get('Contents', []):
    s3_key     = obj['Key']                                          # S3 object key (acts as relative path)
    local_path = os.path.join(LOCAL_DIR, s3_key.replace('/', os.sep))  # Map S3 key to local path

    # Create any intermediate directories that don't exist yet
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    s3.download_file(
        Bucket=BUCKET_NAME,    # Source bucket
        Key=s3_key,            # Object to download
        Filename=local_path    # Local destination path
    )
    print(f"Downloaded: s3://{BUCKET_NAME}/{s3_key} → {local_path}")

Downloaded: s3://kousik-bucket-2026/file4 → C:\projects\aws\Sample\file4
Downloaded: s3://kousik-bucket-2026/folder1/file1 → C:\projects\aws\Sample\folder1\file1
Downloaded: s3://kousik-bucket-2026/folder2/file2 → C:\projects\aws\Sample\folder2\file2
Downloaded: s3://kousik-bucket-2026/folder3/file3 → C:\projects\aws\Sample\folder3\file3
